<a href="https://colab.research.google.com/github/imgeG/CYT-300-Group-1/blob/M4/CYT300M4_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re                     # regular expressions (finding addresses, links, phrases)
import os                     # file paths
import html                   # decoding HTML entities (&amp; -> &)
import numpy as np            # math (log transform)
import pandas as pd           # tables
import joblib                 # saving fitted tools to a file
import kagglehub              # downloading the Kaggle dataset
from functools import lru_cache                 # remembers results of repeated function calls (speed)
from urllib.parse import urlparse               # splits a URL into host, path, etc.
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from scipy.sparse import hstack, csr_matrix, save_npz   # sparse matrices (mostly-zero data)

In [2]:
# Download the dataset (Colab reuses a cached copy if it already exists)
path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")

# Datasets to use; Enron and Ling are excluded because they have no sender/receiver addresses
FILES = {
    "CEAS_08": "CEAS_08.csv",
    "Nazario": "Nazario.csv",
    "Nigerian_Fraud": "Nigerian_Fraud.csv",
    "SpamAssassin": "SpamAssasin.csv",
}

# Only the columns the model uses
COLS = ['sender', 'receiver', 'subject', 'body', 'label']

# Read each file, keep the needed columns, tag each row with its source, and stack them into one table
df = pd.concat([pd.read_csv(os.path.join(path, f))[COLS].assign(source=name)
                for name, f in FILES.items()], ignore_index=True)
print(f"Loaded {len(df)} emails")

Using Colab cache for faster access to the 'phishing-email-dataset' dataset.
Loaded 49860 emails


In [3]:
# =====================================================================
# Cleaning
# =====================================================================
# Columns that together define "the same email" (used for duplicate checks)
KEY = ['sender', 'receiver', 'subject', 'body']

def clean(d):
    """Drop rows with no sender, fill missing text, remove contradictory and duplicate emails."""
    before = len(d)
    d = d.dropna(subset=['sender', 'label']).copy()    # sender and label are required
    for c in KEY:
        d[c] = d[c].fillna('').astype(str)             # missing text becomes an empty string
    d['label'] = d['label'].astype(int)                # 0 = legit, 1 = phishing/spam
    # If the exact same email appears with both labels, we can't know which is right -> drop it
    d = d[d.groupby(KEY)['label'].transform('nunique') == 1]
    d = d.drop_duplicates(subset=KEY)                  # keep one copy of each repeated email
    print(f"clean(): {before} -> {len(d)} rows")
    return d


# =====================================================================
# Address helpers
# =====================================================================
ADDR_RE = re.compile(r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+')   # an email address

# Free webmail providers (attackers often send from these instead of a company domain)
FREE_MAIL = {'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'aol.com',
             'msn.com', 'live.com', 'mail.com', 'yandex.ru', 'gmx.com'}

def first_address(s):
    """First email address found in a field, lowercased ('' if none)."""
    m = ADDR_RE.search(s)
    return m.group(0).lower() if m else ''

def display_name(s):
    """Name shown before the address, e.g. 'PayPal Support <x@y.com>' -> 'PayPal Support'."""
    return s.split('<')[0].strip().strip('"').strip() if '<' in s else ''


# =====================================================================
# Text cleaning (input for TF-IDF)
# =====================================================================
URL_RE   = re.compile(r'https?://\S+|www\.\S+')           # a URL
TAG_RE   = re.compile(r'<[^>]+>')                         # an HTML tag
BLOCK_RE = re.compile(r'(?is)<(script|style).*?</\1>')    # a whole <script>/<style> block

def clean_text(s, max_len=5000):
    s = s[:20000]                          # cap huge bodies before running regex (speed)
    s = BLOCK_RE.sub(' ', s)               # remove script/style contents
    s = TAG_RE.sub(' ', s)                 # remove remaining HTML tags
    s = html.unescape(s)                   # &amp; -> &
    s = URL_RE.sub(' urltoken ', s)        # each link becomes one placeholder word
    s = ADDR_RE.sub(' emailtoken ', s)     # each email address becomes one placeholder word
    return re.sub(r'\s+', ' ', s).strip().lower()[:max_len]   # collapse spaces, lowercase, cap length


# =====================================================================
# Phishing indicators (fixed rules, nothing is learned)
# VirusTotal answers "is this link/domain KNOWN to be bad";
# these rules answer "does this email LOOK deceptive".
# =====================================================================

# Brand -> its real domains. A domain that LOOKS like a brand but isn't one of these is suspicious.
BRANDS = {
    'paypal': {'paypal.com'},
    'microsoft': {'microsoft.com', 'live.com', 'outlook.com', 'office.com', 'office365.com', 'microsoftonline.com'},
    'google': {'google.com', 'gmail.com', 'googlemail.com'},
    'amazon': {'amazon.com', 'amazon.ca', 'amazonaws.com'},
    'apple': {'apple.com', 'icloud.com'},
    'netflix': {'netflix.com'},
    'facebook': {'facebook.com', 'fb.com', 'facebookmail.com'},
    'instagram': {'instagram.com'},
    'linkedin': {'linkedin.com'},
    'dropbox': {'dropbox.com'},
    'docusign': {'docusign.com', 'docusign.net'},
    'fedex': {'fedex.com'},
    'chase': {'chase.com'},
    'wellsfargo': {'wellsfargo.com'},
    'bankofamerica': {'bankofamerica.com'},
    'citibank': {'citibank.com', 'citi.com'},
    'americanexpress': {'americanexpress.com', 'aexp.com'},
    'ebay': {'ebay.com'},
}
MIN_FUZZY_LEN = 5      # skip fuzzy matching for very short brand names (too many false matches)
SUSP_TLDS = {'xyz', 'top', 'click', 'buzz', 'tk', 'ml', 'ga', 'cf', 'gq', 'work', 'link', 'icu',
             'cfd', 'sbs', 'rest', 'zip', 'mov', 'ru', 'cn'}                    # TLDs often abused
SHORTENERS = {'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 'is.gd', 'buff.ly',
              'rebrand.ly', 'cutt.ly', 'tiny.cc'}                              # hide the real destination
TWO_PART_SUFFIX = {'co.uk', 'com.au', 'co.jp', 'com.br', 'co.in', 'org.uk', 'com.cn',
                   'co.za', 'com.mx'}                                          # so evil.co.uk is read correctly

# Character swaps attackers use: 0->o, 1->l, 3->e, etc.
LEET = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '7': 't', '$': 's', '@': 'a'})

def norm_label(s):
    """Undo lookalike tricks: paypa1 -> paypal, rnicrosoft -> microsoft ('rn' looks like 'm')."""
    return s.translate(LEET).replace('rn', 'm').replace('vv', 'w')

def lev(a, b):
    """Levenshtein distance: number of single-character edits to turn a into b (paypall vs paypal = 1)."""
    if abs(len(a) - len(b)) > 2:
        return 3                                   # lengths too different, give up early
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def reg_domain(host):
    """Registrable domain: mail.evil.co.uk -> evil.co.uk, login.paypal.com -> paypal.com."""
    p = host.lower().strip('.').split('.')
    if len(p) >= 3 and '.'.join(p[-2:]) in TWO_PART_SUFFIX:
        return '.'.join(p[-3:])
    return '.'.join(p[-2:]) if len(p) >= 2 else host

@lru_cache(maxsize=None)     # the same domain appears in many emails, so remember results
def domain_risk(host):
    """Four 0/1 flags: typo (paypall.com), leet (paypa1.com),
    brand in domain (paypal-support.com), brand in subdomain (paypal.com.secure-login.ru)."""
    host = host.lower().strip('.')
    reg  = reg_domain(host)
    sld  = reg.split('.')[0]                                            # name part, e.g. 'paypa1-support'
    sub  = host[:-len(reg)].strip('.') if host.endswith(reg) else ''    # anything before the registrable domain
    tokens = {sld} | set(sld.split('-'))                                # also test each hyphen-separated piece
    typo = leet = in_dom = in_sub = 0
    for brand, official in BRANDS.items():
        if reg in official:
            continue                                                    # the real brand domain is not a squat
        for t in tokens:
            if not t:
                continue
            if brand in t:
                in_dom = 1
            elif len(brand) >= MIN_FUZZY_LEN:
                n = norm_label(t)
                if n == brand:
                    leet = 1
                elif lev(n, brand) <= (1 if len(brand) <= 7 else 2):    # 1 edit for short brands, 2 for long
                    typo = 1
        if sub and brand in sub:
            in_sub = 1
    return typo, leet, in_dom, in_sub

def sender_risk(domain, name):
    """Sender impersonation flags, including the display name: 'PayPal Support <x@random.com>'."""
    typo, leet, in_dom, in_sub = domain_risk(domain)
    reg = reg_domain(domain)
    name_key = re.sub(r'[^a-z0-9]', '', name.lower())
    disp = int(any(b in name_key and reg not in off for b, off in BRANDS.items()))
    return {'s_typosquat': typo, 's_leet': leet, 's_brand_in_domain': in_dom,
            's_brand_in_subdomain': in_sub, 's_display_brand_mismatch': disp,
            's_punycode': int('xn--' in domain),                       # internationalized lookalike encoding
            's_nonascii': int(any(ord(c) > 127 for c in domain)),      # non-English characters in the domain
            's_susp_tld': int(domain.split('.')[-1] in SUSP_TLDS)}     # TLD often abused by attackers

RAW_URL_RE = re.compile(r'https?://[^\s<>"\')]+', re.I)                                  # URLs in the raw body
HREF_RE    = re.compile(r'(?is)<a\s[^>]*href\s*=\s*["\']?([^"\'\s>]+)[^>]*>(.*?)</a>')   # (destination, link text)
DOMAINISH  = re.compile(r'\b[a-z0-9-]+(?:\.[a-z0-9-]+)+\b')                              # looks like a domain
IP_RE      = re.compile(r'^\d{1,3}(?:\.\d{1,3}){3}$')                                    # a raw IP address

def host_of(u):
    try:
        return (urlparse(u).hostname or '').lower()
    except ValueError:                             # malformed URL
        return ''
def netloc_of(u):
    """The part of a URL between // and the first / (e.g. 'paypal.com@evil.com'), or '' if malformed."""
    try:
        return urlparse(u).netloc
    except ValueError:                             # malformed URL
        return ''

def url_risk(body):
    """Structural link checks only (VirusTotal handles reputation)."""
    body  = body[:20000]
    urls  = RAW_URL_RE.findall(body)
    hosts = [h for h in (host_of(u) for u in urls) if h]
    out = dict(u_count=len(urls), u_ip=0, u_shortener=0, u_at=0, u_anchor_mismatch=0)
    if not hosts:
        return out
    out['u_ip']        = int(any(IP_RE.match(h) for h in hosts))               # link goes to a bare IP
    out['u_shortener'] = int(any(reg_domain(h) in SHORTENERS for h in hosts))  # link hides its destination
    out['u_at']        = int(any('@' in netloc_of(u) for u in urls))          # http://paypal.com@evil.com trick
    for href, text in HREF_RE.findall(body):                                    # link text shows one domain,
        shown = DOMAINISH.findall(TAG_RE.sub(' ', text).lower())                # but the link goes to another
        h = host_of(href)
        if shown and h and reg_domain(shown[0]) != reg_domain(h):
            out['u_anchor_mismatch'] = 1
            break
    return out

# Phrases that signal pressure, and phrases that ask for credentials/sensitive data
URGENCY_RE = r'\b(?:urgent|immediately|within 24 hours|suspended|final notice|act now|expire[sd]?|limited time|verify your|confirm your)\b'
CRED_RE    = r'\b(?:password|log ?in|sign ?in|username|social security|ssn|account number|credit card|card number|pin)\b'


# =====================================================================
# Feature builder: used for the training data AND the test dataset
# =====================================================================
def build_features(d):
    """Parse addresses and compute address, text and phishing-indicator features."""
    d = d.copy()

    # Extract the sender's address; drop rows where none was found
    d['sender_addr'] = d['sender'].apply(first_address)
    print(f"build_features(): dropped {(d['sender_addr'] == '').sum()} rows with no valid sender address")
    d = d[d['sender_addr'] != ''].copy()

    # Clean subject and body; drop rows with no text left
    d['subject_clean'] = d['subject'].apply(clean_text)
    d['body_clean']    = d['body'].apply(clean_text)
    empty = (d['subject_clean'] + d['body_clean']).str.len() == 0
    print(f"build_features(): dropped {empty.sum()} rows with no text after cleaning")
    d = d[~empty].copy()

    # Split the address into parts: username @ domain, and the TLD
    local              = d['sender_addr'].str.split('@').str[0]
    d['sender_domain'] = d['sender_addr'].str.split('@').str[1]
    d['sender_tld']    = d['sender_domain'].str.split('.').str[-1]
    d['sender_name']   = d['sender'].apply(display_name)

    # All receiver addresses, how many there are, and the first receiver's domain
    receivers  = d['receiver'].apply(lambda s: [a.lower() for a in ADDR_RE.findall(s)])
    rcv_count  = receivers.apply(len)
    rcv_domain = receivers.apply(lambda a: a[0].split('@')[1] if a else 'none')

    # ----- Address features (log1p compresses large values; it means log(1 + x)) -----
    d['sender_local_len_log'] = np.log1p(local.str.len())                    # username length
    d['sender_local_digits']  = local.str.count(r'\d')                       # digits in username
    d['sender_domain_len']    = d['sender_domain'].str.len()                 # domain length
    d['sender_subdomains']    = d['sender_domain'].str.count(r'\.')          # dots in domain
    d['sender_domain_hyphen'] = d['sender_domain'].str.count('-')            # hyphens (lookalike domains)
    d['sender_domain_digits'] = d['sender_domain'].str.count(r'\d')          # digits in domain
    d['sender_free_mail']     = d['sender_domain'].isin(FREE_MAIL).astype(int)        # free webmail sender
    d['sender_has_name']      = (d['sender_name'] != '').astype(int)                  # display name present
    d['receiver_count_log']   = np.log1p(rcv_count)                          # number of recipients
    d['receiver_missing']     = (rcv_count == 0).astype(int)                 # no recipient address
    d['same_domain']          = (d['sender_domain'] == rcv_domain).astype(int)        # internal-looking email
    d['sender_is_receiver']   = [int(a in r) for a, r in zip(d['sender_addr'], receivers)]  # sent to itself

    # ----- Text features (computed on raw text, where capitals and symbols still exist) -----
    full = d['subject'] + ' ' + d['body']
    txt  = d['subject_clean'] + ' ' + d['body_clean']
    d['subject_len_log']     = np.log1p(d['subject_clean'].str.len())        # subject length
    d['body_len_log']        = np.log1p(d['body_clean'].str.len())           # body length
    d['exclaim_count_log']   = np.log1p(full.str.count('!'))                 # exclamation marks
    d['money_symbols_log']   = np.log1p(full.str.count(r'[$£€]'))            # money symbols
    d['upper_ratio']         = d['body'].str.count(r'[A-Z]') / d['body'].str.len().clip(lower=1)        # capitals in body
    d['subject_upper_ratio'] = d['subject'].str.count(r'[A-Z]') / d['subject'].str.len().clip(lower=1)  # capitals in subject
    d['subject_is_reply']    = d['subject'].str.contains(r'(?i)^\s*(?:re|fwd?)\s*:').astype(int)        # starts with Re:/Fw:
    d['kw_urgency_log']      = np.log1p(txt.str.count(URGENCY_RE))           # urgency phrases
    d['kw_cred_log']         = np.log1p(txt.str.count(CRED_RE))              # credential-request phrases

    # ----- Phishing indicators -----
    s_feats = pd.DataFrame([sender_risk(dom, nm) for dom, nm in zip(d['sender_domain'], d['sender_name'])],
                           index=d.index)
    u_feats = pd.DataFrame([url_risk(b) for b in d['body']], index=d.index)
    u_feats['u_count'] = np.log1p(u_feats['u_count'])                        # number of links (log)
    return pd.concat([d, s_feats, u_feats], axis=1)


# All numeric features (address + text + phishing indicators)
NUM_FEATURES = ['sender_local_len_log', 'sender_local_digits', 'sender_domain_len', 'sender_subdomains',
                'sender_domain_hyphen', 'sender_domain_digits', 'sender_free_mail', 'sender_has_name',
                'receiver_count_log', 'receiver_missing', 'same_domain', 'sender_is_receiver',
                'subject_len_log', 'body_len_log', 'exclaim_count_log', 'money_symbols_log', 'upper_ratio',
                'subject_upper_ratio', 'subject_is_reply', 'kw_urgency_log', 'kw_cred_log',
                's_typosquat', 's_leet', 's_brand_in_domain', 's_brand_in_subdomain', 's_display_brand_mismatch',
                's_punycode', 's_nonascii', 's_susp_tld',
                'u_count', 'u_ip', 'u_shortener', 'u_at', 'u_anchor_mismatch']

In [4]:
# Clean the combined data, then compute all features
train = build_features(clean(df))

# Labels the model learns from
y_train = train['label'].values
print(f"Training emails: {len(train)} | phishing ratio: {y_train.mean():.3f}")

# Remove features that never change in this data (they carry no information)
constant = [f for f in NUM_FEATURES if train[f].nunique() <= 1]
NUM_FEATURES = [f for f in NUM_FEATURES if f not in constant]
print("Removed constant features:", constant)

# Report rare indicator features (present in under 0.1% of emails) so you know how much they can help
rare = [f for f in NUM_FEATURES if (train[f] != 0).mean() < 0.001]
print("Rare features (<0.1% of emails):", rare)

clean(): 49860 -> 49529 rows
build_features(): dropped 1019 rows with no valid sender address
build_features(): dropped 0 rows with no text after cleaning
Training emails: 48510 | phishing ratio: 0.559
Removed constant features: ['s_leet', 's_punycode', 's_nonascii', 'u_anchor_mismatch']
Rare features (<0.1% of emails): ['s_typosquat', 's_brand_in_subdomain', 'u_at']


In [5]:
# One-hot encoding of sender TLDs
# Keep the 10 most common TLDs as their own columns; everything else goes into TLD_other
TOP_TLDS = sorted(train['sender_tld'].value_counts().head(10).index)

def encode_tld(d):
    """One 0/1 column per top TLD, plus TLD_other."""
    out = pd.DataFrame({f'TLD_{t}': (d['sender_tld'] == t).astype(int) for t in TOP_TLDS}, index=d.index)
    out['TLD_other'] = (~d['sender_tld'].isin(TOP_TLDS)).astype(int)
    return out


# Sender-domain reputation
# Overall phishing rate, used as the default for domains with little or no history
PRIOR = train['label'].mean()

def reputation_table(d, m=5):
    """Phishing rate per sender domain, smoothed: m=5 pulls domains with few emails toward the average."""
    s = d.groupby('sender_domain')['label'].agg(['sum', 'count'])
    return (s['sum'] + m * PRIOR) / (s['count'] + m)

# Score training rows fold by fold: each fold gets its reputation from the OTHER 4 folds,
# so no email's own label is used to build its own feature (that would be leakage).
# Grouping by sender keeps all of a sender's emails in the same fold.
train['sender_domain_rep'] = np.nan
rep_col = train.columns.get_loc('sender_domain_rep')
for tr_idx, va_idx in GroupKFold(5).split(train, groups=train['sender_addr']):
    table = reputation_table(train.iloc[tr_idx])
    train.iloc[va_idx, rep_col] = train.iloc[va_idx]['sender_domain'].map(table).fillna(PRIOR).values

# Reputation table built from all training data, used later for the test dataset
REP_TABLE = reputation_table(train)


# Combine numeric features
def numeric_matrix(d):
    """Numeric features + TLD columns + reputation, in a fixed column order."""
    return pd.concat([d[NUM_FEATURES], encode_tld(d), d[['sender_domain_rep']]], axis=1)

X_num = numeric_matrix(train)
FEATURES = list(X_num.columns)   # remember the exact column order

# Scaling
# Scale every numeric feature to 0-1 (the same range as TF-IDF values); learned from training data
scaler = MinMaxScaler().fit(X_num)

# TF-IDF (separate vocabularies for subject and body)
# Words that only identify which dataset an email came from (developer mailing lists),
# removed so the model can't use them as a shortcut
CORPUS_WORDS = {'python', 'perl', 'opensuse', 'linux', 'dev', 'mailing', 'list', 'lists', 'unsubscribe',
                'listinfo', 'mailman', 'spamassassin', 'apache', 'sourceforge', 'ceas', 'spam'}
STOP  = list(ENGLISH_STOP_WORDS | CORPUS_WORDS)   # common English words + corpus words
TOKEN = r'(?u)\b[a-z][a-z]+\b'                    # words of 2+ letters only (numbers are ignored)

# Subjects are short with their own vocabulary, so they get their own vectorizer
text_tf = ColumnTransformer([
    ('subj', TfidfVectorizer(ngram_range=(1, 2),      # single words and two-word phrases
                             min_df=3,                # ignore terms in fewer than 3 emails
                             max_features=5000,       # cap vocabulary size
                             sublinear_tf=True,       # dampen repeated words
                             stop_words=STOP, token_pattern=TOKEN), 'subject_clean'),
    ('body', TfidfVectorizer(ngram_range=(1, 2),
                             min_df=5,                # ignore terms in fewer than 5 emails
                             max_df=0.9,              # ignore terms in more than 90% of emails
                             max_features=20000,
                             sublinear_tf=True,
                             stop_words=STOP, token_pattern=TOKEN), 'body_clean'),
]).fit(train)   # learn both vocabularies from the training data

In [6]:
# Final training matrix: scaled numeric features side by side with the TF-IDF word columns
X_train = hstack([csr_matrix(scaler.transform(X_num)),
                  text_tf.transform(train)]).tocsr()

# Name of every column, for interpreting the model later
FULL_FEATURE_NAMES = FEATURES + list(text_tf.get_feature_names_out())
print("X_train:", X_train.shape, "| y_train:", y_train.shape)

# Save to Google Drive so the files survive a runtime reset
from google.colab import drive
drive.mount('/content/drive')
out = '/content/drive/MyDrive/CYT300'
os.makedirs(out, exist_ok=True)

save_npz(f'{out}/X_train.npz', X_train)    # features (sparse format)
np.save(f'{out}/y_train.npy', y_train)     # labels

# Fitted tools, needed to prepare any new data the same way
joblib.dump({
    'scaler': scaler,                                   # fitted 0-1 scaler
    'text_tf': text_tf,                                 # fitted subject + body vocabularies
    'top_tlds': TOP_TLDS,                               # TLDs that get their own column
    'rep_table': REP_TABLE,                             # domain reputation scores
    'prior': PRIOR,                                     # default reputation for unseen domains
    'num_features': NUM_FEATURES,                       # numeric features used (after removing constants)
    'features': FEATURES,                               # numeric column order
    'full_feature_names': FULL_FEATURE_NAMES,           # all column names
    'train_keys': set(pd.util.hash_pandas_object(train[KEY], index=False)),   # fingerprints of training emails
}, f'{out}/preprocessing.joblib')
print("Saved to", out)

X_train: (48510, 25042) | y_train: (48510,)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to /content/drive/MyDrive/CYT300


In [7]:
# Fingerprint of every training email, used to find overlap with the test dataset
TRAIN_KEYS = set(pd.util.hash_pandas_object(train[KEY], index=False))

def prepare_test(raw):
    """Turn a raw test dataset into X_test, y_test using the tools fitted on the training data.
    The dataset needs columns: sender, receiver, subject, body, label."""
    # Same cleaning and features as the training data
    d = build_features(clean(raw[COLS]))

    # Remove emails that are also in the training data, otherwise the test score is inflated
    overlap = pd.util.hash_pandas_object(d[KEY], index=False).isin(list(TRAIN_KEYS)).values
    print(f"prepare_test(): removed {overlap.sum()} emails already in the training data")
    d = d[~overlap].copy()
    if d.empty:
        raise ValueError("No test emails left: every email in this dataset is also in the training data.")

    # Reputation from the training table; domains never seen in training get the average
    d['sender_domain_rep'] = d['sender_domain'].map(REP_TABLE).fillna(PRIOR)

    # Scale with the training scaler and vectorize with the training vocabulary (transform only, no fitting)
    X = hstack([csr_matrix(scaler.transform(numeric_matrix(d)[FEATURES])),
                text_tf.transform(d)]).tocsr()
    return X, d['label'].values

# Example usage:
# test_raw = pd.read_csv('your_test_dataset.csv')
# X_test, y_test = prepare_test(test_raw)
# save_npz(f'{out}/X_test.npz', X_test)
# np.save(f'{out}/y_test.npy', y_test)